In [8]:
!apt-get install -y fonts-nanum


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-nanum is already the newest version (20200506-1).
0 upgraded, 0 newly installed, 0 to remove and 38 not upgraded.


In [9]:
# -*- coding: utf-8 -*-
"""
통합 파이프라인 (수정판)
- 매출 컬럼 자동 탐지 (매출금액 / 매출금액 구간 처리)
- 시계열 파생변수 생성 (rolling, pct_change, trend)
- 조기경보 라벨링(폐업 3~6개월 전)
- VIF 기반 다중공선성 검사/제거
- CoxPH 생존분석 (store-level)
- Supervised 조기경보 분류 (RandomForest, optional XGBoost)
- 한글폰트 설정 및 기본 시각화 출력
"""
import os
import platform
import warnings
warnings.filterwarnings('ignore')

import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정 (환경에 맞게)
import matplotlib.font_manager as fm
if platform.system() == 'Windows':
    plt.rc('font', family='Malgun Gothic')
elif platform.system() == 'Darwin':
    plt.rc('font', family='AppleGothic')
else:
    try:
        plt.rc('font', family='NanumGothic')
    except Exception:
        plt.rc('font', family='sans-serif')
plt.rcParams['axes.unicode_minus'] = False

# ML / survival
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.utils import concordance_index
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

# optional xgboost
try:
    from xgboost import XGBClassifier
    has_xgb = True
except Exception:
    has_xgb = False

# -------------------------
# 설정: 데이터 경로
# -------------------------
DATA_PATH = '/content/drive/MyDrive/GamjaNeverDie/Total_Data/Total_Data_v2.csv'
if not os.path.exists(DATA_PATH):
    DATA_PATH = 'Total_Data_v2.csv'  # 대체 경로

print("1) 데이터 로드 및 날짜 정리 시작")
df = pd.read_csv(DATA_PATH)
print(f"데이터 로드 성공: {df.shape}")

# 날짜 자동 변환
for col in ['기준년월', '개설일', '폐업일']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
if '기준년월' in df.columns:
    df['기준년월'] = df['기준년월'].dt.to_period('M').dt.to_timestamp()

# -------------------------
# 매출 컬럼 자동 탐지 / 구간 -> 수치 변환
# -------------------------
print("2) 매출 컬럼 자동 탐지 및 (구간->수치) 변환")

# 탐지 우선순위: '매출금액' (숫자) -> '매출금액 구간' 등
store_id_col = None
for col in df.columns:
    if col.strip() in ['가맹점구분번호', '가맹점ID', '가맹점번호', 'store_id', 'storeID']:
        store_id_col = col
        break
if store_id_col is None:
    # 시도: 부분 일치
    for col in df.columns:
        if '가맹점' in col and ('구분' in col or 'ID' in col or 'id' in col):
            store_id_col = col
            break

sales_col = None
for col in df.columns:
    if '매출금액' == col:
        sales_col = col
        break
# 매출금액 정확히 없을 경우 '매출금액 구간' 등으로 대체
if sales_col is None:
    for col in df.columns:
        if '매출금액' in col and '구간' in col:
            sales_col = col
            break

if store_id_col is None or sales_col is None:
    raise ValueError(f"필수 컬럼 누락: store_id_col={store_id_col}, sales_col={sales_col}\n현재 컬럼 목록: {list(df.columns)}")

print(f"사용 가맹점 ID 컬럼: {store_id_col}")
print(f"사용 매출 컬럼: {sales_col}")

# helper: 구간 문자열을 숫자로 바꾸는 함수 (가능한 경우 평균값 사용)
def parse_sales_interval_to_number(val):
    """
    Examples of expected formats:
    '100000~200000', '10만원 ~ 20만원', '100,000원 ~ 200,000원', '100000 이상', '50만원 이하'
    Strategy:
      - find all numbers in the string, if two numbers -> mean, if one -> use it * adjustment if '이상' present
      - if no parse, return NaN
    """
    if pd.isna(val):
        return np.nan
    s = str(val)
    # remove commas and currency symbols (원, 만원)
    s_clean = s.replace(',', '').replace('원', '')
    # convert 만원 to numeric * 10000 if present
    # find if '만원' in s
    is_mw = '만원' in s_clean
    # extract numbers
    nums = re.findall(r'[-]?\d+\.?\d*', s_clean)
    if len(nums) == 0:
        return np.nan
    nums = [float(n) for n in nums]
    # adjust for 만원 unit
    if is_mw:
        nums = [n * 10000 for n in nums]
    if len(nums) == 1:
        # check for '이상' or '이하'
        if '이상' in s_clean:
            return nums[0] * 1.2  # 추정치
        elif '이하' in s_clean:
            return nums[0] * 0.8
        else:
            return nums[0]
    else:
        return float(np.mean(nums))

# If the sales_col is categorical (contains non-numeric), try convert
if not np.issubdtype(df[sales_col].dtype, np.number):
    # try converting direct numeric first
    df[sales_col + '_raw'] = df[sales_col]
    df[sales_col] = pd.to_numeric(df[sales_col], errors='coerce')
    # if still many NaNs, try parsing intervals
    if df[sales_col].isna().sum() > 0:
        df[sales_col] = df[sales_col].fillna(df[sales_col + '_raw'].apply(parse_sales_interval_to_number))
    # final fill: if still NaN, leave as NaN (will be handled later)
    df.drop(columns=[sales_col + '_raw'], inplace=True)

# ----------------------------------------------------------------
# 3) 시계열 파생변수 생성 (정렬 필요)
# ----------------------------------------------------------------
print("3) 시계열 파생변수 생성")
df = df.sort_values([store_id_col, '기준년월']).reset_index(drop=True)

# 전월 대비 변화율 (pct_change)
df['매출_전월비_pct'] = df.groupby(store_id_col)[sales_col].pct_change()

# rolling stats
df['매출_3mo_mean'] = df.groupby(store_id_col)[sales_col].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['매출_3mo_std'] = df.groupby(store_id_col)[sales_col].transform(lambda x: x.rolling(3, min_periods=1).std())
df['매출_6mo_mean'] = df.groupby(store_id_col)[sales_col].transform(lambda x: x.rolling(6, min_periods=1).mean())

# 6개월 추세 (선형기울기)
def rolling_trend(series, window=6):
    # returns NaN when insufficient
    def slope(y):
        if np.isnan(y).any() or len(y) < window:
            return np.nan
        return np.polyfit(np.arange(len(y)), y, 1)[0]
    return series.rolling(window, min_periods=window).apply(lambda y: slope(y), raw=True)

df['매출_6mo_trend'] = df.groupby(store_id_col)[sales_col].transform(lambda x: rolling_trend(x, window=6))

# 급감 여부 (전월대비 -20% 기준)
df['급감(-20%_YoY)'] = (df['매출_전월비_pct'] <= -0.2).astype(int)

print("파생변수 생성 완료")

# ----------------------------------------------------------------
# 4) 운영개월(duration) 및 event 정의 (생존분석)
# ----------------------------------------------------------------
print("4) 운영개월 및 event 정의")
df['폐업일_대체'] = df['폐업일'].fillna(df['기준년월'])
df['운영개월'] = ((df['폐업일_대체'].dt.year - df['개설일'].dt.year) * 12 +
                (df['폐업일_대체'].dt.month - df['개설일'].dt.month)).clip(lower=0)

# event: 폐업일 있는 행 & 폐업일 <= 기준년월 => event=1
df['event'] = np.where(df['폐업일'].notna() & (df['폐업일'] <= df['기준년월']), 1, 0)

# ----------------------------------------------------------------
# 5) 조기경보 라벨링 (폐업 3~6개월 전 관측치)
# ----------------------------------------------------------------
print("5) 조기경보 라벨링 (폐업 3~6개월 전)")

df['조기경보'] = 0

# safer group-wise labeling
for store, g in df.groupby(store_id_col):
    closed_dates = g['폐업일'].dropna().unique()
    if len(closed_dates) == 0:
        continue
    close_date = pd.to_datetime(closed_dates[0])
    # mask is a boolean Series with same index as g
    mask = (g['기준년월'] >= (close_date - pd.DateOffset(months=6))) & (g['기준년월'] <= (close_date - pd.DateOffset(months=3)))
    mask_idx = mask[mask].index
    if len(mask_idx) > 0:
        df.loc[mask_idx, '조기경보'] = 1

print("라벨링 완료 - '조기경보' 컬럼 생성 (0/1)")

# ----------------------------------------------------------------
# 6) 피처 후보 선택 및 NaN/inf 처리
# ----------------------------------------------------------------
print("6) 피처 후보 선택 및 결측/무한대 처리")
candidate_features = [
    '매출_전월비_pct', '매출_3mo_mean', '매출_3mo_std', '매출_6mo_mean', '매출_6mo_trend',
    '급감(-20%_YoY)', '배달매출금액 비율', '전월대비 매출금액 감소율(%)',
    '매출 안정성(변동성) CV(3개월)', '재방문 고객 비중_y', '동일 업종 매출건수 비율'
]
# keep only existing
features = [f for f in candidate_features if f in df.columns]
print("초기 사용 피처:", features)

# replace inf and -inf
df = df.replace([np.inf, -np.inf], np.nan)

# df_model for supervised learning: drop rows missing features or target
df_model = df.dropna(subset=features + ['조기경보']).copy()
print(f"Supervised 학습용 rows: {df_model.shape[0]}")

# ----------------------------------------------------------------
# 7) VIF 기반 다중공선성 검사/제거
# ----------------------------------------------------------------
print("7) VIF 계산 및 다중공선성 검사")
if len(features) > 0 and df_model.shape[0] > 0:
    X_vif = df_model[features].copy()
    # in case still NaNs, drop them
    X_vif = X_vif.dropna()
    df_model = df_model.loc[X_vif.index]
    def calculate_vif_df(X):
        vif_data = pd.DataFrame()
        vif_data['feature'] = X.columns
        vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
        return vif_data
    try:
        vif_df = calculate_vif_df(X_vif.fillna(0))
        print(vif_df.sort_values('VIF', ascending=False))
        high_vif = vif_df[vif_df['VIF'] > 10]['feature'].tolist()
        if high_vif:
            print("VIF > 10 컬럼 제거:", high_vif)
            features = [f for f in features if f not in high_vif]
            df_model = df_model.drop(columns=high_vif, errors='ignore')
    except Exception as e:
        print("VIF 계산 오류:", e)
else:
    print("VIF 검사 생략(피처 없음 또는 데이터 부족)")

print("최종 학습 피처:", features)

# ----------------------------------------------------------------
# 8) Supervised 모델 학습 (RandomForest) — 조기경보 예측
# ----------------------------------------------------------------
print("8) Supervised 학습: RandomForest")

if len(features) == 0 or df_model.shape[0] < 50:
    print("학습 불가: 피처 부족 또는 데이터 수 부족")
else:
    X = df_model[features].values
    y = df_model['조기경보'].values
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25,
                                                        random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight='balanced'
    )
    rf.fit(X_train_s, y_train)
    y_pred_rf = rf.predict(X_test_s)

    # 예외 안전한 확률 계산
    if len(rf.classes_) > 1:
        y_proba_rf = rf.predict_proba(X_test_s)[:, 1]
    else:
        y_proba_rf = np.ones(len(X_test_s)) if rf.classes_[0] == 1 else np.zeros(len(X_test_s))

    print("\nRandomForest 결과:")
    print(classification_report(y_test, y_pred_rf))
    print("ROC-AUC:", roc_auc_score(y_test, y_proba_rf))


    # XGBoost (optional)
    if has_xgb:
        xgb = XGBClassifier(n_estimators=200, max_depth=6, use_label_encoder=False, eval_metric='logloss', random_state=42)
        xgb.fit(X_train_s, y_train)
        y_pred_xgb = xgb.predict(X_test_s)
        y_proba_xgb = xgb.predict_proba(X_test_s)[:,1]
        print("\nXGBoost 결과:")
        print(classification_report(y_test, y_pred_xgb, digits=4))
        print("AUC (XGB):", roc_auc_score(y_test, y_proba_xgb))
    else:
        print("\nXGBoost 미설치: 필요시 설치 후 실행하세요.")

    # 변수 중요도 시각화
    if len(features) > 0:
        importances = pd.Series(rf.feature_importances_, index=features).sort_values()
        plt.figure(figsize=(8, max(3, 0.3*len(features))))
        importances.plot(kind='barh')
        plt.title("RandomForest - 변수 중요도")
        plt.xlabel("중요도")
        plt.grid(axis='x', linestyle='--', alpha=0.6)
        plt.tight_layout()
        plt.show()

# ----------------------------------------------------------------
# 9) CoxPH 모델 (store-level)
# ----------------------------------------------------------------
print("9) Cox PH 모델 적합 (store-level)")

last_obs = df.sort_values([store_id_col, '기준년월']).groupby(store_id_col).tail(1).copy()
cox_features = [f for f in features if f in last_obs.columns]

cox_df = last_obs[['운영개월', 'event'] + cox_features].dropna()
if cox_df.shape[0] < 50:
    print("Cox 적합 불가: 충분한 store-level 데이터 없음")
else:
    cph = CoxPHFitter()
    try:
        cph.fit(cox_df, duration_col='운영개월', event_col='event', show_progress=True)
        print("Cox 모델 요약:")
        print(cph.summary)
        c_index = concordance_index(cox_df['운영개월'], -cph.predict_partial_hazard(cox_df), cox_df['event'])
        print("Cox 모델 C-index:", round(c_index, 4))
    except Exception as e:
        print("Cox 적합 오류:", e)

# ----------------------------------------------------------------
# 10) Kaplan-Meier & 급감 여부별 비교
# ----------------------------------------------------------------
print("10) Kaplan-Meier 시각화 (급감 여부별)")

if '급감(-20%_YoY)' in last_obs.columns:
    kmf = KaplanMeierFitter()
    plt.figure(figsize=(8,6))
    for val in [0,1]:
        mask = last_obs['급감(-20%_YoY)'] == val
        if mask.sum() > 0:
            kmf.fit(last_obs.loc[mask, '운영개월'], last_obs.loc[mask, 'event'], label=f"급감={val}")
            kmf.plot_survival_function(ci_show=True)
    plt.title("급감 여부별 생존곡선 (마지막 관측 기준)")
    plt.xlabel("운영 개월")
    plt.ylabel("생존확률")
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()
else:
    print("급감 컬럼 없음: KM 비교 생략")

# ----------------------------------------------------------------
# 11) 실무용: 현재(마지막 관측) 기준 위험 상위 추출
# ----------------------------------------------------------------
print("11) 조기경보 대상 상위 추출 (마지막 관측 기준)")

if 'rf' in locals() and 'scaler' in locals() and len(features) > 0:
    last_X = last_obs[features].dropna()
    if last_X.shape[0] > 0:
        last_X_s = scaler.transform(last_X.values)
        last_obs_prob = rf.predict_proba(last_X_s)[:,1]
        last_obs.loc[last_X.index, '위험확률_RF'] = last_obs_prob
        top_risk = last_obs.sort_values('위험확률_RF', ascending=False).head(20)
        display_cols = [store_id_col, '기준년월', '위험확률_RF'] + features
        print("상위 20개 위험 가맹점 (RandomForest 기준):")
        print(top_risk[display_cols].to_string(index=False))
    else:
        print("마지막 관측 기준 피처 부족으로 위험률 산출 불가")
else:
    print("RandomForest 모델 또는 스케일러 미존재로 위험 상위 산출 불가")

print("\n전체 파이프라인 완료")


1) 데이터 로드 및 날짜 정리 시작
데이터 로드 성공: (86590, 53)
2) 매출 컬럼 자동 탐지 및 (구간->수치) 변환
사용 가맹점 ID 컬럼: 가맹점구분번호
사용 매출 컬럼: 매출금액 구간
3) 시계열 파생변수 생성
파생변수 생성 완료
4) 운영개월 및 event 정의
5) 조기경보 라벨링 (폐업 3~6개월 전)
라벨링 완료 - '조기경보' 컬럼 생성 (0/1)
6) 피처 후보 선택 및 결측/무한대 처리
초기 사용 피처: ['매출_전월비_pct', '매출_3mo_mean', '매출_3mo_std', '매출_6mo_mean', '매출_6mo_trend', '급감(-20%_YoY)', '배달매출금액 비율', '전월대비 매출금액 감소율(%)', '매출 안정성(변동성) CV(3개월)', '재방문 고객 비중_y', '동일 업종 매출건수 비율']
Supervised 학습용 rows: 57267
7) VIF 계산 및 다중공선성 검사
                feature        VIF
1           매출_3mo_mean  88.892875
3           매출_6mo_mean  81.442171
4          매출_6mo_trend   7.263902
2            매출_3mo_std   2.278805
0            매출_전월비_pct   1.949970
5          급감(-20%_YoY)   1.879437
6             배달매출금액 비율   1.647191
8   매출 안정성(변동성) CV(3개월)   1.581112
10        동일 업종 매출건수 비율   1.394247
9           재방문 고객 비중_y   1.164717
7      전월대비 매출금액 감소율(%)   1.113142
VIF > 10 컬럼 제거: ['매출_3mo_mean', '매출_6mo_mean']
최종 학습 피처: ['매출_전월비_pct', '매출_3mo_std', '매출_6mo_trend', '급감(-2

XGBoostError: [11:42:42] /workspace/src/objective/./regression_loss.h:68: Check failed: base_score > 0.0f && base_score < 1.0f: base_score must be in (0,1) for logistic loss, got: 0
Stack trace:
  [bt] (0) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x2a6e7c) [0x79c93b4a6e7c]
  [bt] (1) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0xeda699) [0x79c93c0da699]
  [bt] (2) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x6826d3) [0x79c93b8826d3]
  [bt] (3) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x682a9c) [0x79c93b882a9c]
  [bt] (4) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(+0x68cfeb) [0x79c93b88cfeb]
  [bt] (5) /usr/local/lib/python3.12/dist-packages/xgboost/lib/libxgboost.so(XGBoosterUpdateOneIter+0x77) [0x79c93b3b6f57]
  [bt] (6) /lib/x86_64-linux-gnu/libffi.so.8(+0x7e2e) [0x79c9a4107e2e]
  [bt] (7) /lib/x86_64-linux-gnu/libffi.so.8(+0x4493) [0x79c9a4104493]
  [bt] (8) /usr